# $Rocio$ $Argote$ $Hermosilla$

### Rut: 22.373.783-8

In [16]:
import numpy as np
import pandas as pd
import csv
import os
import pickle

### PASO 1: revisar que los archivos esten subidos correctamente.

In [17]:
print(os.getcwd()) #muestra la carpeta donde python esta trabajando
print(os.listdir()) #muestra los archivos dentro de esa carpeta

/drive/notebooks/FISICA COMPUTACIONAL
['Tarea5.FCII.ipynb', 'catalogo_estelar.csv', 'resultados']


In [18]:
with open("catalogo_estelar.csv", "r") as archivo: #en esta linea se abre el archivo, r de read
    lectura = csv.reader(archivo) 
    archivo_csv = next(lectura)  #next obtiene la primera fila del archivo correspondiente
    print(archivo_csv)

['id', 'nombre', 'temperatura_K', 'luminosidad_Lsol', 'distancia_pc']


### PASO 2: crear una clase para almacenamiento de datos del catalogo estelar, limpieza de datos y un resumen

In [1]:
class CatalogoEstelar:
    def __init__(self, ruta_csv): #__init__ es como el constructor de la clase, self representa al objeto especifico que se crea
        datos_catalogo = pd.read_csv(ruta_csv) #abre el archivo en la ruta_csv, lee las columnas, las filas y devuelve una tabla
        self.id = datos_catalogo["id"].to_numpy() #to_numpy convertira cada columna de pandas en un arreglo de numpy
        self.nombre = datos_catalogo["nombre"].to_numpy()
        self.temperatura_K = datos_catalogo["temperatura_K"].to_numpy() 
        self.luminosidad_Lsol = datos_catalogo["luminosidad_Lsol"].to_numpy()
        self.distancia_pc = datos_catalogo["distancia_pc"].to_numpy()
        self.limpiar_datos()


    
#PARTE 1 funcion para limpiar datos
    def limpiar_datos(self):
        #junto las columnas que tienen datos numéricos
        columnas_num = [self.temperatura_K, self.luminosidad_Lsol, self.distancia_pc]
        
        #compruebo que las columnas numéricas no tengan valores ausentes ni infinitos
        columnas_invalidas = [self.temperatura_K, self.luminosidad_Lsol, self.distancia_pc]
        
        #si el id es numérico, también lo incluyo en la comprobacion
        if np.issubdtype(self.id.dtype, np.number):
            columnas_num.append(self.id)

        #mascara donde todas las estrellas se consideran validas
        datos_finitos = np.ones(len(self.id), dtype=bool)

        #compruebo los valores de cada columna numerica
        for columna in columnas_num:
            datos_finitos &= pd.notna(columna) & np.isfinite(columna)

        # Compruebo que los id no esten ausentes
        id_presentes = pd.notna(self.id)

        #reviso los id duplicados
        # duplicated() marca las apariciones posteriores
        datos_repetidos = pd.Series(self.id).duplicated().to_numpy()

        #veo que las propiedades físicas sean positivas
        datos_positivos = ((self.temperatura_K > 0) & (self.luminosidad_Lsol > 0) & (self.distancia_pc > 0))

        #combino todas las condiciones para identificar los datos validos
        datos_validos_totales = ( datos_finitos & id_presentes & ~datos_repetidos & datos_positivos )

        #clloco la misma máscara a todas las columnas para tener misma correspondencia entre los datos
        self.id = self.id[datos_validos_totales]
        self.nombre = self.nombre[datos_validos_totales]
        self.temperatura_K = self.temperatura_K[datos_validos_totales]
        self.luminosidad_Lsol = self.luminosidad_Lsol[datos_validos_totales]
        self.distancia_pc = self.distancia_pc[datos_validos_totales]

    

    
#PARTE 2 creo un resumen de todo
    def resumen(self):    
        total_estrellas = len(self.id) #dira la cantidad total de estrellas
        temperatura_media = np.mean(self.temperatura_K) #media de temperaturas
        temperatura_maxima = np.max(self.temperatura_K) #valor maximo de temperaturas

    
        luminosidad_mediana = np.median(self.luminosidad_Lsol) #media de luminosidades
        distancia_minima = np.min(self.distancia_pc) #valor minimo
        distancia_maxima = np.max(self.distancia_pc) #valor maximo

        posicion_estrella_mas_caliente = np.argmax(self.temperatura_K) #dira el numero de posicion en la que se encuentra
        posicion_estrella_mas_cercana = np.argmin(self.distancia_pc) #numero de posicion de la estrella mas cercana
        #busco primero su posicion antes de su nombre

        print("❀     ─┄───┄───┄───┄───┄───┄───┄───┄┄───╮")
        print("            RESUMEN ANÁLISIS DE DATOS")
        print()
        print("  - Cantidad de estrellas: ", total_estrellas)
        print("  - Temperatura media: ", temperatura_media ,"K")
        print("  - Temperatura máxima: ", temperatura_maxima, "K")
        print("  - Luminosidad mediana: ", luminosidad_mediana, "Luminosidad solar")
        print("  - Distancia mínima: ", distancia_minima, "pc")
        print("  - Distancia máxima: ", distancia_maxima, "pc")
        print("  - Estrella más caliente: ", self.nombre[posicion_estrella_mas_caliente])
        print("  - Estrella más cercana: ", self.nombre[posicion_estrella_mas_cercana])
        print()
        print("╰───┄───┄───┄───┄───┄───┄───┄───┄       °❀")

#PARTE 3 guardo los datos en un csv
    def guardar_csv(self):
        os.makedirs("resultados", exist_ok = True) #os.makedirs crea una carpeta y exist_ok es por si ya existe
        datos = {"id": self.id, "nombre": self.nombre, "temperatura_K": self.temperatura_K, "luminosidad_Lsol": self.luminosidad_Lsol, "distancia_pc": self.distancia_pc}
        #paso los datos a una tabla
        tabla = pd.DataFrame(datos)
        tabla.to_csv("resultados/catalogo_copia.csv", index=False)

    
#PARTE 4 guardo las temperaturas en npy
    def guardar_npy(self):
        #crear la carpeta
        os.makedirs("resultados", exist_ok=True)

        #guardo el arreglo de temperatura
        np.save("resultados/temperaturas.npy", self.temperatura_K)


#PARTE 5 guardo los datos en npz
    def guardar_npz(self):
        os.makedirs("resultados", exist_ok=True)
        np.savez_compressed("resultados/catalogo_completo.npz", id=self.id.astype(str), nombre=self.nombre.astype(str),  temperatura_K=self.temperatura_K, luminosidad_Lsol=self.luminosidad_Lsol, distancia_pc=self.distancia_pc)

    
#PARTE 6 guardo el catálogo en pickle
    def guardar_pickle(self):
        #creo la carpeta
        os.makedirs("resultados", exist_ok=True)

        #junto todas las columnas en un diccionario
        datos = {"id": self.id, "nombre": self.nombre, "temperatura_K": self.temperatura_K, "luminosidad_Lsol": self.luminosidad_Lsol, "distancia_pc": self.distancia_pc}

        #Guardo el diccionario
        with open("resultados/catalogo_completo.pkl", "wb") as archivo:
            pickle.dump(datos, archivo)

#PARTE 7 verifico que los archivos se hayan guardado bien
    def verificar_archivos(self):

        #recupero los datos del archivo CSV
        catalogo_csv = CatalogoEstelar("resultados/catalogo_copia.csv")

        #comparo los datos del CSV con los originales
        csv_correcto = (np.array_equal(self.id, catalogo_csv.id) and np.array_equal(self.nombre, catalogo_csv.nombre) and
            np.allclose(self.temperatura_K, catalogo_csv.temperatura_K) and np.allclose(self.luminosidad_Lsol, catalogo_csv.luminosidad_Lsol) and
            np.allclose(self.distancia_pc, catalogo_csv.distancia_pc) )

        # recupero las temperaturas del archivo NPY
        temperaturas = np.load("resultados/temperaturas.npy")

        # compruebo si coinciden con las originales
        npy_correcto = np.allclose(self.temperatura_K, temperaturas)

        # recupero los datos del archivo NPZ
        catalogo_npz = np.load("resultados/catalogo_completo.npz")

        # comparo las columnas del NPZ con las originales
        npz_correcto = (np.array_equal(self.id, catalogo_npz["id"]) and np.array_equal(self.nombre, catalogo_npz["nombre"]) and np.allclose(self.temperatura_K, catalogo_npz["temperatura_K"]) and
            np.allclose(self.luminosidad_Lsol, catalogo_npz["luminosidad_Lsol"]) and np.allclose(self.distancia_pc, catalogo_npz["distancia_pc"]) )

        # recupero el diccionario guardado en Pickle
        with open("resultados/catalogo_completo.pkl", "rb") as archivo:
            catalogo_pickle = pickle.load(archivo)

        # comparo las columnas del Pickle con las originales
        pickle_correcto = (np.array_equal(self.id, catalogo_pickle["id"]) and np.array_equal(self.nombre, catalogo_pickle["nombre"]) and np.allclose(self.temperatura_K, catalogo_pickle["temperatura_K"]) and
            np.allclose(self.luminosidad_Lsol, catalogo_pickle["luminosidad_Lsol"]) and np.allclose(self.distancia_pc, catalogo_pickle["distancia_pc"]))

        # muestro los resultados de las comprobaciones
        print("CSV:", csv_correcto)
        print("NPY:", npy_correcto)
        print("NPZ:", npz_correcto)
        print("Pickle:", pickle_correcto)

        
# PARTE 8 comparo el tamaño de los archivos
    def comparar_formatos(self):

        # guardo los nombres de los archivos
        archivos = [ "catalogo_copia.csv", "temperaturas.npy", "catalogo_completo.npz", "catalogo_completo.pkl" ]

        # calculo el tamaño de cada archivo en bytes
        tamaño = [ os.path.getsize("resultados/" + archivo) for archivo in archivos ]

        # creo una tabla con los resultados
        tabla = pd.DataFrame({ "Archivo": archivos, "Tamaño (bytes)": tamaño })
        print(tabla)


catalogo_estelar = CatalogoEstelar("catalogo_estelar.csv")

<class 'NameError'>: name 'pd' is not defined

In [20]:
catalogo_estelar.resumen()

❀     ─┄───┄───┄───┄───┄───┄───┄───┄┄───╮
            RESUMEN ANÁLISIS DE DATOS

  - Cantidad de estrellas:  92
  - Temperatura media:  6388.595148307613 K
  - Temperatura máxima:  9829.356461457292 K
  - Luminosidad mediana:  6.027373975844869 Luminosidad solar
  - Distancia mínima:  58.02655278964448 pc
  - Distancia máxima:  490.80680626396673 pc
  - Estrella más caliente:  Estrella_0006
  - Estrella más cercana:  Estrella_0039

╰───┄───┄───┄───┄───┄───┄───┄───┄       °❀


In [21]:
catalogo_estelar.guardar_csv()

In [22]:
catalogo_estelar.guardar_npy()

In [23]:
catalogo_estelar.guardar_npz()

In [24]:
catalogo_estelar.guardar_pickle()

In [25]:
print(os.listdir("resultados"))

['catalogo_completo.npz', 'catalogo_completo.pkl', 'catalogo_copia.csv', 'temperaturas.npy']


In [26]:
#tamaño de cada archivo
for archivo in os.listdir("resultados"): 
    ruta = "resultados/" + archivo 
    tamaño = os.path.getsize(ruta)
    print(archivo, ":", tamaño, "bytes")

catalogo_completo.npz : 3809 bytes
catalogo_completo.pkl : 4813 bytes
catalogo_copia.csv : 6922 bytes
temperaturas.npy : 864 bytes


In [27]:
#leo nuevamente el archivo csv usando la clase
catalogo_csv = CatalogoEstelar("resultados/catalogo_copia.csv")

catalogo_csv.resumen()

❀     ─┄───┄───┄───┄───┄───┄───┄───┄┄───╮
            RESUMEN ANÁLISIS DE DATOS

  - Cantidad de estrellas:  92
  - Temperatura media:  6388.595148307613 K
  - Temperatura máxima:  9829.356461457292 K
  - Luminosidad mediana:  6.027373975844869 Luminosidad solar
  - Distancia mínima:  58.02655278964448 pc
  - Distancia máxima:  490.80680626396673 pc
  - Estrella más caliente:  Estrella_0006
  - Estrella más cercana:  Estrella_0039

╰───┄───┄───┄───┄───┄───┄───┄───┄       °❀


In [28]:
# recupero el arreglo de temperaturas que guardé
temperaturas_recuperadas = np.load("resultados/temperaturas.npy")

# muestro las primeras tres temperaturas para comprobar los datos
print(temperaturas_recuperadas[:3])

[8417.69233989 6072.14907826 9010.18543938]


In [29]:
# recupero todos los datos que guardé en el archivo npz
catalogo_npz = np.load("resultados/catalogo_completo.npz")

# muestro las primeras tres temperaturas recuperadas
print(catalogo_npz["temperatura_K"][:3])

[8417.69233989 6072.14907826 9010.18543938]


In [30]:
# recupero el diccionario que guardé en pickle
with open("resultados/catalogo_completo.pkl", "rb") as archivo:
    catalogo_pickle = pickle.load(archivo)

# muestro las primeras trestemperaturas recuperadas
print(catalogo_pickle["temperatura_K"][:3])

[8417.69233989 6072.14907826 9010.18543938]


In [31]:
# comparo las temperaturas recuperadas con las originales
print(np.array_equal(temperaturas_recuperadas, catalogo_estelar.temperatura_K))

# comparo las temperaturas del archivo npz con las originales
print(np.array_equal(catalogo_npz["temperatura_K"], catalogo_estelar.temperatura_K))

# comparo las temperaturas del archivo pickle con las originales
print(np.array_equal(catalogo_pickle["temperatura_K"], catalogo_estelar.temperatura_K))
#yeyy

True
True
True


In [32]:
#creo una lista para guardar los nombres y tamaños de mis archivos
archivos = os.listdir("resultados")

#guardo los nombres y tamaños en una tabla
comparacion = pd.DataFrame({"archivo": archivos, "tamaño_bytes": [os.path.getsize("resultados/" + archivo)for archivo in archivos]})

print(comparacion)

                 archivo  tamaño_bytes
0  catalogo_completo.npz          3809
1  catalogo_completo.pkl          4813
2     catalogo_copia.csv          6922
3       temperaturas.npy           864


In [33]:
catalogo_estelar.verificar_archivos()

CSV: True
NPY: True
NPZ: True
Pickle: True


In [34]:
catalogo_estelar.comparar_formatos()

                 Archivo  Tamaño (bytes)
0     catalogo_copia.csv            6922
1       temperaturas.npy             864
2  catalogo_completo.npz            3809
3  catalogo_completo.pkl            4813


### comparacion de formatos:

csv deja guardar y compartir datos fácilmente, npy guarda un solo arreglo de numpy, npz permite guardar varios arreglos comprimidos en un archivo, y pickle permite guardar objetos de python. npy me ocupó menos espacio porque solo guardó las temperaturas, mientras que npz fue el más liviano entre los formatos que guardaban el catálogo completo.


### 1. ¿qué formato utilizaría para compartir el catálogo con otros usuarios?

Usaria el formato csv, ya que te permite guardar los datos en forma de tabla, facilitando su entendimiento y se puede abrir de distintas maneras, y además más sencillo de compartir distintos datos 

### 2. ¿cuál considera más apropiado para almacenar varios arreglos NumPy en un único archivo?

Usaria el formato csv, porque me he familiarizado más con él y me resulta más sencillo de utilizar, pero tengo entendido que el csv a diferencia de npz, no está diseñado específicamente para almacenar varios arreglos numpy en un único archivo